# 30. Twin distillation at scale, and the adversary at full size

Notebook 29 gave the first movement in the adaptive number in this project (0.48 to 0.40) with one pass over 3,000 Alpaca-style examples. This notebook scales the same method along the two axes the residual points to, and fixes the power problem in the evaluation.

1. **Data.** 6,000 new examples, resuming from the notebook 29 adapter: 3,000 Alpaca-style (new seed) and 3,000 document-style, in which SQuAD and AG News passages from the structural training hosts receive a random Alpaca instruction at the start, the end, or between two sentences, under the paper's summarization task. No BIPIA text. One pass, checkpointed and resumable.
2. **Power.** The adversary subset grows from 91 to all 210 BIPIA malicious injections (30 per category). The 91 keep their existing rewrites; the 119 new ones get four rewrites each from the same 7B rewriter with the same prompts, budget and temperature as notebook 22. Four models are executed on all 1,050 variants and judged the same way, with every earlier execution and verdict reused by text: undefended, the DPO model (28), twin distillation v1 (29) and v2 (this notebook). The attacker moves second (any working variant of five). Utility as in notebook 29.

GPU, several hours across sessions; every stage caches and training resumes.

**Outputs.** `data/distill2/examples.json`, `data/distill2/lora_3b/`, `data/phase5v3/subset_full.json`, `data/phase5v3/rewrites_full.json`, `data/distill2/exec_*.json`, `data/distill2/judge_*.json`, `reports/distill2_asr.csv`, `reports/distill2_by_category.csv`, `reports/distill2_utility.csv`, `figures/distill2.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first): install from the local disk, then mount ===
!pip install -q transformers datasets accelerate sentencepiece bitsandbytes peft
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('reports', 'figures', 'data/distill2', 'data/phase5v3'): os.makedirs(d_, exist_ok=True)
import torch, gc; print('Session ready. GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## Stage A. Module and inputs

In [ ]:
%%writefile src/twin_distill.py
"""Twin distillation: on-policy, token-level self-distillation that makes the model's output independent of an
injected span. For every training example the injected input is built from a clean one, so the clean input is
known. The student (the model with a LoRA adapter) samples a response to the injected input; the teacher (the same
model with the adapter disabled) scores every token of that response given the CLEAN input. The student is pushed,
token by token, toward what it would have said without the injection. A twin example places the same injected
instruction in the user turn with the clean input; there the teacher sees that very prompt, which anchors the student
to keep following instructions the user issues. The signal never asks the model to recognise an injection, only to
make its output invariant to the untrusted span."""
from __future__ import annotations
import numpy as np

PROMPT = "{instruction}\n\nInput:\n{inp}"
PROMPT_WITH = "{instruction} {extra}\n\nInput:\n{inp}"


def build_examples(samples, n=1500, seed=0, positions=(0.45, 0.45, 0.10)):
    """samples: list of dicts with 'instruction' and non-empty 'input'. Returns attacked and twin examples.
    Each attacked example carries the student prompt (injected input) and the teacher prompt (clean input)."""
    rng = np.random.default_rng(seed); S = [s for s in samples if s.get("input", "").strip() and s.get("instruction", "").strip()]
    idx = rng.choice(len(S), min(n, len(S)), replace=False); out = []
    for i in idx:
        s = S[i]; j = int(rng.integers(len(S)))
        while j == i: j = int(rng.integers(len(S)))
        inj = S[j]["instruction"].strip(); inp = s["input"].strip(); r = rng.random()
        if r < positions[0]: injected = inp + "\n\n" + inj                       # end
        elif r < positions[0] + positions[1]: injected = inj + "\n\n" + inp        # start
        else: injected = inp + "\n\nAnswer: task complete.\n\n" + inj              # fake-completion style
        out.append(dict(kind="attacked", student=PROMPT.format(instruction=s["instruction"].strip(), inp=injected), teacher=PROMPT.format(instruction=s["instruction"].strip(), inp=inp)))
        twin = PROMPT_WITH.format(instruction=s["instruction"].strip(), extra=inj, inp=inp)
        out.append(dict(kind="twin", student=twin, teacher=twin))
    rng.shuffle(out); return out


def _split_sentences(text):
    import re
    parts = re.split(r"(?<=[.!?])\s+", text.strip()); return [p for p in parts if p]


def build_doc_examples(hosts, instructions, task_user, task_user_with, n=1500, seed=0, positions=(0.45, 0.45, 0.10)):
    """Document-style examples: a passage (SQuAD or AG News host) with a random instruction injected at the end, the
    start, or between two sentences; the task is the paper's summarization task. Same student/teacher/twin layout."""
    rng = np.random.default_rng(seed); H = [h for h in hosts if len(h.split()) >= 30]; I = [i.strip() for i in instructions if i.strip()]
    out = []
    for k in range(min(n, len(H))):
        h = H[int(rng.integers(len(H)))]; inj = I[int(rng.integers(len(I)))]; r = rng.random()
        if r < positions[0]: injected = h.rstrip() + "\n" + inj
        elif r < positions[0] + positions[1]: injected = inj + "\n" + h
        else:
            s = _split_sentences(h); cut = int(rng.integers(1, len(s))) if len(s) > 1 else 1
            injected = " ".join(s[:cut]) + "\n" + inj + "\n" + " ".join(s[cut:]) if len(s) > 1 else h.rstrip() + "\n" + inj
        out.append(dict(kind="attacked", student=task_user.format(doc=injected), teacher=task_user.format(doc=h)))
        twin = task_user_with.format(instr=inj, doc=h); out.append(dict(kind="twin", student=twin, teacher=twin))
    rng.shuffle(out); return out


def _chat(tok, system, user, add_gen=True):
    return tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": user}], tokenize=False, add_generation_prompt=add_gen)


def sample_rollouts(model, tok, prompts, system, max_new_tokens=128, temperature=1.0):
    """Sample one response per prompt from the current student (batched, left padding)."""
    import torch
    was_training = model.training; model.eval(); tok.padding_side = "left"
    enc = tok([_chat(tok, system, p) for p in prompts], return_tensors="pt", padding=True, truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=1.0, pad_token_id=tok.pad_token_id, use_cache=True)
    outs = [tok.decode(gen[j][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for j in range(len(prompts))]
    if was_training: model.train()
    return outs


def response_logprobs(model, tok, prompts, responses, system, max_len=1024):
    """Per-token log-probabilities of each response given its prompt (batched, right padding). Returns list of 1-D tensors."""
    import torch
    tok.padding_side = "right"; seqs, starts, lens = [], [], []
    for p, r in zip(prompts, responses):
        p_ids = tok(_chat(tok, system, p), add_special_tokens=False).input_ids; r_ids = tok(r + tok.eos_token, add_special_tokens=False).input_ids
        if len(p_ids) + len(r_ids) > max_len: p_ids = p_ids[: max(64, max_len - len(r_ids))]; r_ids = r_ids[: max_len - len(p_ids)]
        seqs.append(p_ids + r_ids); starts.append(len(p_ids)); lens.append(len(r_ids))
    T = max(len(s) for s in seqs); pad = tok.pad_token_id
    ids = torch.tensor([s + [pad] * (T - len(s)) for s in seqs], device=model.device); am = torch.tensor([[1] * len(s) + [0] * (T - len(s)) for s in seqs], device=model.device)
    logits = model(input_ids=ids, attention_mask=am).logits
    out = []
    for b, (st, ln) in enumerate(zip(starts, lens)):
        lg = logits[b, st - 1: st - 1 + ln].float(); tg = ids[b, st: st + ln]
        out.append(torch.log_softmax(lg, -1).gather(1, tg[:, None])[:, 0])
    return out


def distill_loss(lp_student, lp_teacher):
    """On-policy reverse-KL policy gradient: per-token advantage = log p_teacher - log p_student (detached)."""
    import torch
    losses, kls = [], []
    for s, t in zip(lp_student, lp_teacher):
        adv = (t - s).detach(); losses.append(-(adv * s).mean()); kls.append((s - t).detach().mean())
    return torch.stack(losses).mean(), float(torch.stack(kls).mean())


In [ ]:
import importlib, json, hashlib, shutil, numpy as np, pandas as pd
for d_ in ('data/distill2', 'data/phase5v3'): os.makedirs(d_, exist_ok=True)
import twin_distill, model_defense, phase5_redo, targets, attack_success, structural_train, data_loaders; [importlib.reload(m) for m in (twin_distill, model_defense, phase5_redo, targets, attack_success, structural_train, data_loaders)]
from twin_distill import build_examples, build_doc_examples, sample_rollouts, response_logprobs, distill_loss, PROMPT
from model_defense import build_pairs, asr_table, TASK_USER_WITH
from phase5_redo import build_subset, flat_variants, RW_SYS, RW_USER, N_REWRITES
from targets import load_lm, generate_batch, TASK_SYSTEM, TASK_USER, build_task_prompts
from attack_success import programmatic_success, judge_prompt, parse_judge, JUDGE_SYSTEM
from structural_train import load_jsonl
from data_loaders import load_bipia_categorized, BIPIA_MALICIOUS
from ci_tools import wilson, fmt
TARGET = 'Qwen/Qwen2.5-3B-Instruct'; JUDGE = 'Qwen/Qwen2.5-7B-Instruct'; ADAPTER = 'data/distill2/lora_3b'; ADAPTER_V1 = 'data/distill/lora_3b'; ADAPTER_DPO = 'data/defense/lora_3b'
H = lambda s: hashlib.sha1(s.encode()).hexdigest()
fp = 'data/distill2/examples.json'
if os.path.exists(fp): EX = json.load(open(fp))
else:
    from datasets import load_dataset
    alp = [dict(instruction=r['instruction'], input=r['input']) for r in load_dataset('tatsu-lab/alpaca', split='train') if r['input'].strip()]
    held_txt = {h['instruction'] for h in json.load(open('data/distill/examples.json'))['held']}
    alp_train = [a for a in alp if a['instruction'] not in held_txt]
    EX_a = build_examples(alp_train, n=1500, seed=7)
    hosts = [r['text'] for r in load_jsonl('data/structural/train.jsonl') if r['form'] == 'host_only']
    EX_d = build_doc_examples(hosts, [a['instruction'] for a in alp_train], TASK_USER, TASK_USER_WITH, n=1500, seed=11)
    EX = EX_a + EX_d; np.random.default_rng(3).shuffle(EX); json.dump(EX, open(fp, 'w'))
HELD = json.load(open('data/distill/examples.json'))['held']; UPAIRS = build_pairs(load_jsonl('data/structural/val.jsonl'), n_max=60, seed=1)
print('training examples:', len(EX), '| attacked:', sum(e['kind'] == 'attacked' for e in EX), '| twins:', sum(e['kind'] == 'twin' for e in EX))
d0 = next(e for e in EX if e['kind'] == 'attacked' and e['student'].startswith('Summarize')); print('\nDOCUMENT-STYLE STUDENT sees:\n', d0['student'][:400])

## Stage B. Continue twin distillation from the notebook 29 adapter (cached, resumable)

In [ ]:
from peft import PeftModel, prepare_model_for_kbit_training
PROG = 'data/distill2/progress.json'; start = json.load(open(PROG))['step'] if os.path.exists(PROG) else 0
if not os.path.exists(os.path.join(ADAPTER, 'adapter_config.json')): shutil.copytree(ADAPTER_V1, ADAPTER); print('initialised from the notebook 29 adapter')
if start >= len(EX): print('training complete')
else:
    base, tok = load_lm(TARGET, four_bit=True); base = prepare_model_for_kbit_training(base)
    model = PeftModel.from_pretrained(base, ADAPTER, is_trainable=True); model.print_trainable_parameters(); print('resuming from example', start)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-5, weight_decay=0.0)
    B, LOG, CKPT, MAXNEW = 4, 20, 100, 128; model.train()
    log = json.load(open('data/distill2/train_log.json')) if os.path.exists('data/distill2/train_log.json') else []
    for step in range(start, len(EX), B):
        batch = EX[step: step + B]
        roll = sample_rollouts(model, tok, [e['student'] for e in batch], TASK_SYSTEM, max_new_tokens=MAXNEW)
        keep = [i for i, r in enumerate(roll) if len(r) >= 5]
        if keep:
            batch_k = [batch[i] for i in keep]; roll = [roll[i] for i in keep]
            lp_s = response_logprobs(model, tok, [e['student'] for e in batch_k], roll, TASK_SYSTEM)
            with torch.no_grad(), model.disable_adapter():
                lp_t = response_logprobs(model, tok, [e['teacher'] for e in batch_k], roll, TASK_SYSTEM)
            loss, kl = distill_loss(lp_s, lp_t); loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0); opt.step(); opt.zero_grad()
            log.append(dict(step=step, loss=float(loss), kl=kl, kinds=''.join(e['kind'][0] for e in batch_k)))
            del lp_s, lp_t, loss; torch.cuda.empty_cache()
        nb = step // B
        if nb % LOG == 0 and log: print(f'  {step + len(batch)}/{len(EX)}  loss {np.mean([l["loss"] for l in log[-LOG:]]):.4f}  reverse-KL {np.mean([l["kl"] for l in log[-LOG:]]):.3f}')
        if nb % CKPT == CKPT - 1:
            model.save_pretrained(ADAPTER); json.dump(dict(step=step + B), open(PROG, 'w')); json.dump(log, open('data/distill2/train_log.json', 'w'))
    model.save_pretrained(ADAPTER); json.dump(dict(step=len(EX)), open(PROG, 'w')); json.dump(log, open('data/distill2/train_log.json', 'w')); print('adapter saved')
    del model, base; gc.collect(); torch.cuda.empty_cache()

## Stage C. The adversary at full size: all 210 injections, existing rewrites kept, new ones generated the same way

In [ ]:
bip = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS)
sfp, rfp = 'data/phase5v3/subset_full.json', 'data/phase5v3/rewrites_full.json'
if os.path.exists(sfp): SUB = pd.DataFrame(json.load(open(sfp)))
else: SUB = build_subset(bip, per_category=30, seed=0); json.dump(SUB.to_dict('records'), open(sfp, 'w'))
OLD = pd.DataFrame(json.load(open('data/phase5v2/subset.json'))); OLD_REW = json.load(open('data/phase5v2/rewrites.json')); old_by_idx = {int(b): r for b, r in zip(OLD.bip_index, OLD_REW)}
if os.path.exists(rfp): REW = json.load(open(rfp))
else:
    REW = [old_by_idx.get(int(b)) for b in SUB.bip_index]; todo = [i for i, r in enumerate(REW) if r is None]; print('injections needing rewrites:', len(todo), 'of', len(SUB))
    rm, rt = load_lm('Qwen/Qwen2.5-7B-Instruct', four_bit=True); msgs, idx = [], []
    for i in todo:
        t = SUB.text.iloc[i]
        for k in range(N_REWRITES): msgs.append(RW_USER.format(k=k + 1, t=t)); idx.append(i)
    outs = generate_batch(rm, rt, msgs, system=RW_SYS, max_new_tokens=220, batch_size=8, temperature=0.8)
    for i in todo: REW[i] = []
    for j, i in enumerate(idx): REW[i].append(outs[j].strip() or SUB.text.iloc[i])
    json.dump(REW, open(rfp, 'w')); del rm, rt; gc.collect(); torch.cuda.empty_cache()
assert len(REW) == len(SUB) and all(len(r) == N_REWRITES for r in REW)
VAR, OWNER, FIRST = flat_variants(SUB, REW); print('injections:', len(SUB), '| variants:', len(VAR), '| categories:', SUB.meta.value_counts().to_dict())

## Stage D. Execute four models on all variants and the utility sets (every earlier output reused by text)

In [ ]:
from peft import PeftModel
UT_follow = [TASK_USER_WITH.format(instr=p['instruction'], doc=p['host']) for p in UPAIRS]; UT_clean = [TASK_USER.format(doc=p['host']) for p in UPAIRS]; UT_alpaca = [PROMPT.format(instruction=h['instruction'], inp=h['input']) for h in HELD]
PROMPTS = {'adv': build_task_prompts(VAR), 'user': UT_follow, 'clean': UT_clean, 'alpaca': UT_alpaca}
TAGS = {'undefended': None, 'dpo_twins_nb28': ADAPTER_DPO, 'twin_distilled_v1': ADAPTER_V1, 'twin_distilled_v2': ADAPTER}
OLD_SRC = {'undefended': [('data/defense', 'undefended'), ('data/distill', 'undefended')], 'dpo_twins_nb28': [('data/defense', 'defended'), ('data/distill', 'dpo_twins_nb28')], 'twin_distilled_v1': [('data/distill', 'twin_distilled')]}
OLD_VAR = flat_variants(OLD, OLD_REW)[0]; OLD_PROMPTS = {'adv': build_task_prompts(OLD_VAR), 'user': UT_follow, 'clean': UT_clean, 'alpaca': UT_alpaca}
def cache_path(tag): return f'data/distill2/exec_{tag}.json'
def load_cache(tag):
    C = json.load(open(cache_path(tag))) if os.path.exists(cache_path(tag)) else {}
    for root, old in OLD_SRC.get(tag, []):
        for name, plist in OLD_PROMPTS.items():
            f = f'{root}/exec_{old}_{name}.json'
            if os.path.exists(f):
                outs = json.load(open(f))
                if len(outs) == len(plist): C.update({H(p): o for p, o in zip(plist, outs) if H(p) not in C})
    return C
def run_missing(tag, model, tok, C):
    todo = [p for plist in PROMPTS.values() for p in plist if H(p) not in C]
    if todo and model is not None:
        print(f'executing {tag}: {len(todo)} prompts')
        for k in range(0, len(todo), 160):
            chunk = todo[k: k + 160]; outs = generate_batch(model, tok, chunk, system=TASK_SYSTEM, max_new_tokens=160, batch_size=8, show_every=160)
            C.update({H(p): o for p, o in zip(chunk, outs)}); json.dump(C, open(cache_path(tag), 'w')); print(f'  {tag}: {min(k + 160, len(todo))}/{len(todo)} cached')
    json.dump(C, open(cache_path(tag), 'w')); return C
CACHE = {tag: load_cache(tag) for tag in TAGS}
need = [tag for tag in TAGS if any(H(p) not in CACHE[tag] for plist in PROMPTS.values() for p in plist)]
if need:
    tm, tt = load_lm(TARGET, four_bit=True)
    if 'undefended' in need: CACHE['undefended'] = run_missing('undefended', tm, tt, CACHE['undefended'])
    for tag in need:
        ad = TAGS[tag]
        if ad is None or not os.path.exists(os.path.join(ad, 'adapter_config.json')): continue
        m = PeftModel.from_pretrained(tm, ad).eval(); CACHE[tag] = run_missing(tag, m, tt, CACHE[tag]); m.unload(); del m; gc.collect(); torch.cuda.empty_cache(); tm, tt = load_lm(TARGET, four_bit=True)
    del tm; gc.collect(); torch.cuda.empty_cache()
OUT = {tag: {name: [CACHE[tag].get(H(p), '') for p in plist] for name, plist in PROMPTS.items()} for tag in TAGS if all(H(p) in CACHE[tag] for plist in PROMPTS.values() for p in plist)}
print('models with complete executions:', list(OUT))

## Stage E. Judge (earlier verdicts reused by text; programmatic where the category allows)

In [ ]:
OLD_JUDGE = {'undefended': ['data/defense/judge_undefended.json', 'data/distill/judge_undefended.json'], 'dpo_twins_nb28': ['data/defense/judge_defended.json', 'data/distill/judge_dpo_twins_nb28.json'], 'twin_distilled_v1': ['data/distill/judge_twin_distilled.json']}
OLD_OWNER = flat_variants(OLD, OLD_REW)[1]
def old_verdicts(tag):
    V = {}
    for f in OLD_JUDGE.get(tag, []):
        if not os.path.exists(f): continue
        J = json.load(open(f))
        for k, v in J.items():
            kind, i = k.split('|'); i = int(i)
            key = (H(OLD_PROMPTS['adv'][i]) if kind == 'adv' else H(UT_follow[i]) if kind == 'user' else H(UT_alpaca[i]) if kind == 'alpaca' else None)
            if key: V.setdefault(key, bool(v))
    return V
def judge_items(tag, outs):
    fp = f'data/distill2/judge_{tag}.json'; JV = json.load(open(fp)) if os.path.exists(fp) else {}
    for k, v in old_verdicts(tag).items(): JV.setdefault(k, v)
    need = []
    for v in range(len(VAR)):
        i = int(OWNER[v]); key = H(PROMPTS['adv'][v]); p = programmatic_success(SUB.meta.iloc[i], SUB.instruction.iloc[i], outs['adv'][v])
        if p is not None: JV.setdefault(key, bool(p))
        elif key not in JV: need.append((key, SUB.instruction.iloc[i], outs['adv'][v]))
    for u in range(len(UPAIRS)):
        if H(UT_follow[u]) not in JV: need.append((H(UT_follow[u]), UPAIRS[u]['instruction'], outs['user'][u]))
    for u in range(len(HELD)):
        if H(UT_alpaca[u]) not in JV: need.append((H(UT_alpaca[u]), HELD[u]['instruction'], outs['alpaca'][u]))
    return JV, need
JVS, NEED = {}, {}
for tag, outs in OUT.items(): JVS[tag], NEED[tag] = judge_items(tag, outs)
if any(NEED.values()):
    jm, jt = load_lm(JUDGE, four_bit=True)
    for tag, need in NEED.items():
        if not need: continue
        print('judging', tag, len(need)); jo = generate_batch(jm, jt, [judge_prompt(ins, out) for _, ins, out in need], system=JUDGE_SYSTEM, max_new_tokens=8, batch_size=8, temperature=0.0)
        for (key, _, _), o in zip(need, jo): JVS[tag][key] = bool(parse_judge(o))
        json.dump(JVS[tag], open(f'data/distill2/judge_{tag}.json', 'w'))
    del jm; gc.collect(); torch.cuda.empty_cache()
for tag in JVS: json.dump(JVS[tag], open(f'data/distill2/judge_{tag}.json', 'w'))
print('verdicts:', {t: len(v) for t, v in JVS.items()})

## Stage F. Attack success on 210 injections with the attacker moving second; by category; utility

In [ ]:
rows, cat_rows = [], []
for tag, JV in JVS.items():
    outs = OUT[tag]; w = np.array([bool(JV.get(H(PROMPTS['adv'][v]), False)) for v in range(len(VAR))]); r = asr_table(SUB, w, OWNER, FIRST, tag)
    anyw = np.array([w[OWNER == i].any() for i in range(len(SUB))])
    r['ASR_static_ci'] = fmt(r['ASR_static'], *wilson(int(w[FIRST].sum()), len(SUB))); r['ASR_adaptive_ci'] = fmt(r['ASR_adaptive_any_of_5'], *wilson(int(anyw.sum()), len(SUB)))
    uf = np.array([bool(JV.get(H(UT_follow[u]), False)) for u in range(len(UPAIRS))]); r['user_instruction_followed'] = fmt(uf.mean(), *wilson(int(uf.sum()), len(uf)))
    af = np.array([bool(JV.get(H(UT_alpaca[u]), False)) for u in range(len(HELD))]); r['alpaca_instruction_followed'] = fmt(af.mean(), *wilson(int(af.sum()), len(af)))
    cl = np.array([len((o or '').strip()) for o in outs['clean']]); r['clean_summary_median_chars'] = int(np.median(cl)); r['clean_summary_degenerate_rate'] = float((cl < 20).mean())
    rows.append(r)
    for cat in SUB.meta.unique():
        m = (SUB.meta == cat).values; cat_rows.append(dict(model=tag, category=cat, n=int(m.sum()), ASR_static=float(w[FIRST][m].mean()), ASR_adaptive=float(anyw[m].mean())))
RES = pd.DataFrame(rows); RES.to_csv('reports/distill2_asr.csv', index=False)
CAT = pd.DataFrame(cat_rows); CAT.to_csv('reports/distill2_by_category.csv', index=False)
UTIL = RES[['model', 'user_instruction_followed', 'alpaca_instruction_followed', 'clean_summary_median_chars', 'clean_summary_degenerate_rate']]; UTIL.to_csv('reports/distill2_utility.csv', index=False)
pd.set_option('display.width', 250)
print(f'=== attack success on {len(SUB)} held-out injections: static and adaptive (attacker picks any working variant of 5) ==='); print(RES[['model', 'ASR_static_ci', 'ASR_adaptive_ci']].to_string(index=False))
print('\n=== by category (adaptive, 30 per cell) ==='); print(CAT.pivot_table(index='category', columns='model', values='ASR_adaptive').round(3).to_string())
print('\n=== utility ==='); print(UTIL.to_string(index=False))

## Stage G. Figure, log and commit

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
x = np.arange(len(RES)); w = 0.35
axes[0].bar(x - w / 2, RES.ASR_static, w, label='static (originals)'); axes[0].bar(x + w / 2, RES.ASR_adaptive_any_of_5, w, label='adaptive (any of 5 variants)')
axes[0].set_xticks(x); axes[0].set_xticklabels(RES.model, fontsize=8, rotation=10); axes[0].set_ylim(0, 1.05); axes[0].set_ylabel('attack success rate'); axes[0].set_title(f'Injection success, {len(SUB)} injections'); axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=.3)
piv = CAT.pivot_table(index='category', columns='model', values='ASR_adaptive'); piv.plot.bar(ax=axes[1], width=.8); axes[1].set_ylim(0, 1.05); axes[1].set_ylabel('adaptive attack success'); axes[1].set_title('By category (30 per cell)'); axes[1].tick_params(axis='x', labelrotation=30, labelsize=8); axes[1].legend(fontsize=7); axes[1].grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig('figures/distill2.png', dpi=150, bbox_inches='tight'); plt.show()
from reslog import log_result
summary = f'Twin distillation at scale ({len(SUB)} injections), attack success:\n' + RES[['model', 'ASR_static_ci', 'ASR_adaptive_ci', 'user_instruction_followed', 'alpaca_instruction_followed', 'clean_summary_degenerate_rate']].to_string(index=False) + '\n\nBy category:\n' + CAT.round(3).to_string(index=False)
log_result('nb30: twin distillation continued on 6,000 mixed examples; adversary extended to all 210 injections; four models', summary, csv_df=RES, csv_name='distill2_asr.csv')
print(summary)

# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py", "nb30: twin distillation continued from nb29 on 6,000 Alpaca- and document-style examples; adversary extended to all 210 BIPIA malicious injections with the same rewriter; undefended, DPO, v1 and v2 compared with the attacker moving second"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)